# Clean the skincare data

In [14]:
import os, re
import numpy as np
import pandas as pd

RAW_PATH          = "combined_filled (1).csv"
PRODUCT_INFO_PATH = "product_info.csv"
CLEAN_PATH        = "skincare_clean.csv"

PKR_PER_USD     = 277.17   # 1 USD = 277.17 PKR
MIN_INGREDIENTS = 3      # fewer = sets/bundles/bad scrape ("no info", "visit the dior boutique")
MAX_INGREDIENTS = 100    # more = a kit with several products mixed together

steps = []
def log(step, frame):
    steps.append((step, len(frame)))
    print(f"{step:<62} {len(frame):>6} rows")

## 1. Load

In [15]:
raw = pd.read_csv(RAW_PATH)
df = raw.copy()
log("Raw data", df)
print(df["source"].value_counts().to_string())

Raw data                                                         3299 rows
source
kaggle (sephora 2023)    1926
kaggle (sephora)         1068
brand site                224
shopify shop               81


## 2. Brand names
Kaggle rows write brands in CAPITALS, the brand sites in normal case. Each brand gets one spelling.

In [16]:
df["brand"] = df["brand"].str.strip()
df["brand"] = df["brand"].str.replace(r"\s+since\s+\d{4}$", "", regex=True, flags=re.I)

def pretty(name):
    return name.title().replace("'S", "'s")

spelling = {}
for key, group in df.groupby(df["brand"].str.lower())["brand"]:
    mixed = [b for b in group if not b.isupper()]
    spelling[key] = pd.Series(mixed).value_counts().index[0] if mixed else pretty(group.iloc[0])
df["brand"] = df["brand"].str.lower().map(spelling)
print(df["brand"].nunique(), "unique brands")

183 unique brands


## 3. Ingredients
Same cleaning as version 1: lowercase, remove labels / percentages / `*` / trailing full stops, keep `1,2-hexanediol` in one piece, 
merge `aqua/water/eau` -> `water` and `parfum` -> `fragrance`. Stored in `ingredients_clean` separated by `|`.

In [17]:
WATER_RE     = re.compile(r"(aqua|water|eau)([ /(]+(aqua|water|eau)\)?)*")
FRAGRANCE_RE = re.compile(r"(fragrance|parfum)([ /(]+(fragrance|parfum)\)?)*")

def clean_ingredients(text):
    if pd.isna(text):
        return []
    t = re.sub(r"\s+", " ", str(text).lower())
    t = re.split(r"may contain|\+/-", t)[0]
    t = re.sub(r"(inactive|active) ingredients?\s*:", ",", t)
    t = re.sub(r"[^,]*\bingredients?\s*:", ",", t)
    t = re.sub(r"\d+(\.\d+)?\s*%", "", t)
    t = re.sub(r"\b(\d),\s*(\d)-", r"\1~\2-", t)              # protect "1, 2-hexanediol" (~ restored below)
    parts = re.split(r"[;,]\s*(?![^()]*\))", t)
    out = []
    for p in parts:
        p = p.replace("*", "").replace("~", ",")
        p = re.sub(r"\s+", " ", p).strip(" .-\u2013")
        if not p or len(p) > 60 or not re.search(r"[a-z]", p):
            continue
        if WATER_RE.fullmatch(p):
            p = "water"
        elif FRAGRANCE_RE.fullmatch(p):
            p = "fragrance"
        out.append(p)
    return list(dict.fromkeys(out))

df = df.dropna(subset=["ingredients"]).copy()
log("Dropped rows with no ingredient list", df)

df["ingredient_list"]   = df["ingredients"].apply(clean_ingredients)
df["ingredient_count"]  = df["ingredient_list"].str.len()
print("Too short:", (df["ingredient_count"] < MIN_INGREDIENTS).sum(), "| too long:", (df["ingredient_count"] > MAX_INGREDIENTS).sum())
df = df[df["ingredient_count"].between(MIN_INGREDIENTS, MAX_INGREDIENTS)].copy()
df["ingredients_clean"] = df["ingredient_list"].apply("|".join)
log(f"Kept products with {MIN_INGREDIENTS}-{MAX_INGREDIENTS} ingredients", df)

Dropped rows with no ingredient list                             3178 rows
Too short: 3 | too long: 9
Kept products with 3-100 ingredients                             3166 rows


## 4. Categories
Raw data has many names for the same thing, some junk (`en-pk`, `Gift Sets`) and some missing values.
We map them to 7 groups. If the category is missing or unclear (`For Face`, or the shop name used as category), 
we look at the **product name** instead (e.g. "Acne Serum" -> Serum).

In [18]:
DROP_RE = re.compile(r"en-pk|catalog|\b(sets?|kits?|duo|trio|gift|bundle|collections?|baby|body|lips?|hair|scalp|"
                     r"shampoo|conditioner|foundation|lash|brow|mascara|supplements?|diabetic|birthday)\b|items")
RULES = [
    (r"\beyes?\b",                                                  "Eye care"),
    (r"cleans|face wash|wipes?\b|makeup remover|micellar",          "Cleanser"),
    (r"moistur|cream|lotion",                                       "Moisturizer"),
    (r"serum|ampoule|hyaluronic|niacinamide|vitamin c|retinol|peptide", "Serum"),
    (r"\bsun|\bspf\b",                                              "Sun protection"),
    (r"mask",                                                       "Mask"),
    (r"treat|exfoliat|peel|acne|aging|barrier|multi|toner|essence|\boils?\b|ointment|blemish", "Treatment"),
]

def category_from_text(text):
    t = str(text).lower()
    if DROP_RE.search(t):
        return "DROP"
    for pattern, label in RULES:
        if re.search(pattern, t):
            return label
    return None

def map_category(cat, name):
    first = category_from_text(cat) if pd.notna(cat) else None
    if first == "DROP":
        return None
    if first:
        return first
    second = category_from_text(name)          # category missing / unclear -> use the name
    return None if second == "DROP" else second

before = df["category"].nunique()
df["category"] = [map_category(c, n) for c, n in zip(df["category"], df["name"])]
df = df.dropna(subset=["category"]).copy()
print(before, "raw categories ->", df["category"].nunique(), "clean categories")
log("Mapped categories (name used when missing), dropped non-face items", df)
print(df["category"].value_counts().to_string())

74 raw categories -> 7 clean categories
Mapped categories (name used when missing), dropped non-face items   2973 rows
category
Moisturizer       658
Treatment         540
Cleanser          454
Serum             389
Mask              365
Eye care          336
Sun protection    231


## 5. Price: one column in USD and one in PKR
- The raw column is called `price_usd`, but the Pakistani-shop rows actually hold **PKR** amounts (`currency` says which). 
- We make both columns for every row: `price_usd` and `price_pkr`, using the exchange rate set in the first cell (`PKR_PER_USD`).
- Rows without a price are dropped so KNN and K-Means never see missing values.
- `price_capped` limits the top 1% of USD prices (scaling is done later in the modeling notebook, on training data only).

In [19]:
price_in_original_currency = pd.to_numeric(df["price_usd"], errors="coerce")
is_pkr = df["currency"].eq("PKR")

df["currency_original"] = df["currency"]
df["price_usd"] = np.where(is_pkr, price_in_original_currency / PKR_PER_USD, price_in_original_currency).round(2)
df["price_pkr"] = np.where(is_pkr, price_in_original_currency, price_in_original_currency * PKR_PER_USD).round(0)

df = df[df["price_usd"] > 0].copy()
log("Dropped rows with missing price", df)

cap = df["price_usd"].quantile(0.99)
df["price_capped"] = df["price_usd"].clip(upper=cap)
print(f"Exchange rate used: 1 USD = {PKR_PER_USD} PKR | USD price cap = ${cap:.0f}")
print(df.groupby("source")[["price_usd", "price_pkr"]].median().round(0).to_string())

Dropped rows with missing price                                  2929 rows
Exchange rate used: 1 USD = 277.17 PKR | USD price cap = $290
                       price_usd  price_pkr
source                                     
brand site                  16.0     4544.0
kaggle (sephora 2023)       48.0    13304.0
kaggle (sephora)            42.0    11641.0
shopify shop                 7.0     2075.0


## 6. Skin-type tags, specificity and "general" products

Each product gets six 0/1 tags: `tag_dry, tag_oily, tag_combination, tag_normal, tag_sensitive, tag_acne_prone` 
(`all` means all five skin types).

For the ranking:
- `n_tags` = how many tags the product has.
- `specificity = 1 / n_tags`: a product tagged only `oily` has 1.0, one tagged for 4 types has 0.25.
- `is_general = 1` when the product is tagged for **all five** skin types. These are recommended **after** the targeted products.
- Products with no skin-type information stay in the data (`has_skin_label = 0`) and are used by KNN and K-Means. 
  We do **not** guess their labels from ingredients, because the Decision Tree learns from ingredients and would just copy our guess.

In [20]:
TYPES = ["dry", "oily", "combination", "normal", "sensitive"]
TAGS  = TYPES + ["acne-prone"]

def parse_tags(s):
    if pd.isna(s) or not str(s).strip():
        return None
    parts = {p.strip() for p in str(s).lower().split("|")}
    if "all" in parts:
        parts |= set(TYPES)
    parts &= set(TAGS)
    return parts or None

parsed = df["skin_types"].apply(parse_tags)
for t in TAGS:
    df["tag_" + t.replace("-", "_")] = parsed.apply(lambda p: np.nan if p is None else int(t in p)).astype("Int64")

tag_cols = ["tag_" + t.replace("-", "_") for t in TAGS]
df["has_skin_label"] = parsed.notna().astype(int)
df["n_tags"]      = df[tag_cols].sum(axis=1).astype("Int64")
df["specificity"] = (1 / df["n_tags"].astype(float)).where(df["has_skin_label"] == 1)
df["is_general"]  = (df[["tag_" + t for t in TYPES]].sum(axis=1) == 5).astype(int)

print("labelled:", df["has_skin_label"].sum(), "| unlabelled:", (df["has_skin_label"] == 0).sum(), "| general (all 5 types):", df["is_general"].sum())
print(df.loc[df["has_skin_label"] == 1, tag_cols].sum().to_string())

labelled: 1644 | unlabelled: 1285 | general (all 5 types): 546
tag_dry            1103
tag_oily            985
tag_combination    1274
tag_normal         1292
tag_sensitive       627
tag_acne_prone      422


## 7. Duplicates
Now that tags exist we can keep the **most complete** copy of a duplicate (has skin tags, has site-written `helps_with`, has rating). 
1. same brand + same name, then 2. same brand + identical ingredient list (same formula in another size or set).

In [21]:
df["_complete"] = (df["has_skin_label"] * 4 + df["helps_with_source"].eq("site") * 2
                   + df["rating"].notna() * 1)
df = df.sort_values("_complete", ascending=False, kind="stable")

df = df.drop_duplicates(subset=["brand", "name"]).copy()
log("Removed same brand + same name", df)
df = df.drop_duplicates(subset=["brand", "ingredients_clean"]).copy()
log("Removed same brand + identical ingredient list", df)
df = df.drop(columns="_complete").sort_index()

Removed same brand + same name                                   2929 rows
Removed same brand + identical ingredient list                   2724 rows


## 8. `helps_with` flags (only real, site-written values)
`helps_with` was written by the website for some rows and **guessed by our scraper from the ingredients** for the others (`helps_with_source = derived`). 
The guessed ones use almost the same rules as the ingredient flags below, so using them would count the same evidence twice. 
The flags below are therefore `1` only when the **site** wrote the concern.

In [22]:
site = df["helps_with_source"].eq("site")
text = df["helps_with"].fillna("").str.lower()

GROUPS = {
    "helps_oil_control": r"oil control|visible shine",
    "helps_acne":        r"acne|blemish|congestion",
    "helps_dryness":     r"dryness|dehydration|hydration|barrier",
    "helps_redness":     r"redness",
}
for col, pattern in GROUPS.items():
    df[col] = (site & text.str.contains(pattern)).astype(int)

print("rows with site-written helps_with:", site.sum(), "of", len(df))
print(df[list(GROUPS)].sum().to_string())

rows with site-written helps_with: 1271 of 2724
helps_oil_control      3
helps_acne           163
helps_dryness        442
helps_redness        104


## 9. Ingredient flags (what the ranking rules look for)

| Skin type | Rank UP | Rank DOWN |
|---|---|---|
| Oily / acne-prone | niacinamide, salicylic acid, zinc, clay, oil-free | heavy oils and butters |
| Dry | ceramides, hyaluronic acid, squalane | - |
| Sensitive | - | fragrance, alcohol denat, essential oils |

Zinc = zinc PCA / gluconate / sulfate (not zinc oxide, which is a sunscreen filter). PEG-castor-oil emulsifiers are not counted as heavy oils. Heavy oils/butters = any butter (shea, cocoa, mango ...), coconut / olive / castor / avocado / palm / mineral oil, petrolatum, lanolin. 
Essential oils = oils from lavender, mint, citrus, eucalyptus, tea tree, rosemary, ylang-ylang, clove, cinnamon, geranium, jasmine, sandalwood, cedar, juniper, thyme, plus menthol and camphor. 
Fatty alcohols (cetyl, stearyl) are **not** counted as alcohol.

**Oil-free** comes from Sephora's own "Oil Free" tag (read from `product_info.csv`) or the words "oil-free" in the product name. 
Products from other sources have no oil-free information, so they stay `0` (unknown).

In [23]:
def any_token(pattern):
    rx = re.compile(pattern)
    return lambda toks: int(any(rx.search(t) for t in toks))

HEAVY_OIL_WORDS = r"coconut|cocos nucifera|olive|olea europaea|avocado|persea|castor|ricinus|palm|elaeis|cocoa|theobroma|mineral|paraffinum"
def heavy_oil_butter(toks):
    for t in toks:
        if re.match(r"(peg|ppg)-", t):          # PEG-40 hydrogenated castor oil is an emulsifier, not a heavy oil
            continue
        if re.search(r"\bbutter\b|petrolatum|lanolin", t):
            return 1
        if re.search(r"\boil\b", t) and re.search(HEAVY_OIL_WORDS, t):
            return 1
    return 0

ESSENTIAL_WORDS = (r"lavandula|lavender|mentha|peppermint|spearmint|eucalyptus|citrus|orange|lemon|bergamot|grapefruit|"
                   r"lime|melaleuca|tea tree|rosmarinus|rosemary|cananga|ylang|syzygium|clove|cinnamomum|cinnamon|"
                   r"pelargonium|geranium|jasminum|jasmine|santalum|sandalwood|cedrus|cedar|juniperus|thymus|thyme")
def essential_oil(toks):
    return int(any((re.search(r"\boil\b", t) and re.search(ESSENTIAL_WORDS, t)) or re.search(r"^(menthol|camphor)$", t)
                   for t in toks))

FLAGS = {
    "has_niacinamide":      any_token(r"niacinamide"),
    "has_salicylic_acid":   any_token(r"salicylic acid|betaine salicylate"),
    "has_zinc":             any_token(r"^zinc( pca| gluconate| sulfate| citrate| aspartate)?$"),   # not zinc oxide (sunscreen) or zinc stearate (texture)
    "has_clay":             any_token(r"kaolin|bentonite|montmorillonite|\bclay\b|fuller'?s earth|rhassoul"),
    "has_heavy_oil_butter": heavy_oil_butter,
    "has_ceramide":         any_token(r"ceramide"),
    "has_hyaluronic":       any_token(r"hyaluronic|hyaluronate"),
    "has_squalane":         any_token(r"squalane|squalene"),
    "has_fragrance":        any_token(r"^(fragrance|perfume)\b"),
    "has_alcohol_denat":    any_token(r"alcohol denat|denatured alcohol|^sd alcohol|^alcohol$|^ethanol$|isopropyl alcohol"),
    "has_essential_oil":    essential_oil,
}
for col, fn in FLAGS.items():
    df[col] = df["ingredient_list"].apply(fn)

# oil-free label
df["oil_free"] = df["name"].str.contains(r"oil[- ]free", case=False).astype(int)
if os.path.exists(PRODUCT_INFO_PATH):
    pi = pd.read_csv(PRODUCT_INFO_PATH, usecols=["brand_name", "product_name", "highlights"])
    pi["oil_free"] = pi["highlights"].fillna("").str.contains("Oil Free")
    key = lambda b, n: b.astype(str).str.lower().str.strip() + "|" + n.astype(str).str.lower().str.strip()
    oil_free_keys = set(key(pi.loc[pi["oil_free"], "brand_name"], pi.loc[pi["oil_free"], "product_name"]))
    df["oil_free"] = (df["oil_free"] | key(raw.loc[df.index, "brand"], df["name"]).isin(oil_free_keys)).astype(int)
else:
    print("product_info.csv not found -> oil_free comes from the product name only")

flag_cols = list(FLAGS) + ["oil_free"]
print((df[flag_cols].mean() * 100).round(1).astype(str).add(" %").to_string())

has_niacinamide         14.0 %
has_salicylic_acid       9.9 %
has_zinc                 6.5 %
has_clay                 7.1 %
has_heavy_oil_butter    26.6 %
has_ceramide             9.3 %
has_hyaluronic          45.2 %
has_squalane            22.2 %
has_fragrance           32.2 %
has_alcohol_denat       18.4 %
has_essential_oil       21.3 %
oil_free                 3.3 %


## 10. Save the clean file

In [24]:
final_cols = (["brand", "name", "category", "ingredients_clean", "ingredient_count",
               "price_usd", "price_pkr", "price_capped"]
              + tag_cols + ["n_tags", "specificity", "is_general", "has_skin_label"]
              + flag_cols + list(GROUPS)
              + ["helps_with", "helps_with_source", "source"])
clean = df[final_cols].reset_index(drop=True)
clean.to_csv(CLEAN_PATH, index=False)
log("FINAL clean file", clean)
print("Saved", CLEAN_PATH, clean.shape)

FINAL clean file                                                 2724 rows
Saved skincare_clean.csv (2724, 37)


## 11. Summary table

In [25]:
summary = pd.DataFrame(steps, columns=["Step", "Rows left"])
summary["Rows removed"] = -summary["Rows left"].diff().fillna(0).astype(int)
summary

,Step,Rows left,Rows removed
0,Raw data,3299,0
1,Dropped rows with no ingredient list,3178,121
2,Kept products with 3-100 ingredients,3166,12
3,"Mapped categories (name used when missing), dr...",2973,193
4,Dropped rows with missing price,2929,44
5,Removed same brand + same name,2929,0
6,Removed same brand + identical ingredient list,2724,205
7,FINAL clean file,2724,0


## 12. Sanity checks

In [26]:
print("Missing values in model columns:")
print(clean[["brand", "category", "ingredients_clean", "price_capped"]].isna().sum().to_string())

vocab = pd.Series([i for lst in clean["ingredients_clean"].str.split("|") for i in lst])
print("\nUnique ingredients:", vocab.nunique(), "| appear only once:", (vocab.value_counts() == 1).sum())
print("'1,2-hexanediol' kept whole:", (vocab == "1,2-hexanediol").sum(), "| broken '2-hexanediol':", (vocab == "2-hexanediol").sum())

print("\nPakistani-shop products kept:", (clean["source"] == "shopify shop").sum())
print("\nRows per source after cleaning:")
print(clean["source"].value_counts().to_string())
print("\nRows per category:")
print(clean["category"].value_counts().to_string())

Missing values in model columns:
brand                0
category             0
ingredients_clean    0
price_capped         0

Unique ingredients: 7316 | appear only once: 3619
'1,2-hexanediol' kept whole: 503 | broken '2-hexanediol': 0

Pakistani-shop products kept: 12

Rows per source after cleaning:
source
kaggle (sephora 2023)    1632
kaggle (sephora)          997
brand site                 83
shopify shop               12

Rows per category:
category
Moisturizer       588
Treatment         492
Cleanser          406
Serum             364
Mask              337
Eye care          320
Sun protection    217


## Column guide (all 37 columns)

**Identity**
| Column | Meaning |
|---|---|
| `brand`, `name` | brand and product name |
| `category` | one of 7 groups: Moisturizer, Treatment, Cleanser, Serum, Mask, Eye care, Sun protection |
| `source` | where the row came from (kaggle sephora, kaggle sephora 2023, brand site, shopify shop) |

**Ingredients**
| Column | Meaning |
|---|---|
| `ingredients_clean` | cleaned ingredient list, ingredients separated by `\|` |
| `ingredient_count` | number of ingredients in that list |

**Price**
| Column | Meaning |
|---|---|
| `price_usd`, `price_pkr` | price in dollars and in rupees (converted with `PKR_PER_USD`) |
| `price_capped` | `price_usd` with the top 1% of prices limited, so expensive products do not dominate distances |

**Skin-type information** (empty when the data has no skin-type information for that product)
| Column | Meaning |
|---|---|
| `tag_dry`, `tag_oily`, `tag_combination`, `tag_normal`, `tag_sensitive`, `tag_acne_prone` | 1 = product is tagged for this skin type, 0 = it is not |
| `has_skin_label` | 1 = the product has skin-type information, 0 = it has none (so all six tags are empty) |
| `n_tags` | how many tags the product has |
| `specificity` | 1 / `n_tags`: 1.0 = made for one skin type, small = made for many |
| `is_general` | 1 = tagged for all five skin types (dry, oily, combination, normal, sensitive) |

**Ingredient flags** (1 = the ingredient group is in the product, 0 = it is not)
| Column | Meaning |
|---|---|
| `has_niacinamide`, `has_salicylic_acid`, `has_zinc`, `has_clay` | oil-control / acne ingredients |
| `has_heavy_oil_butter` | heavy oils and butters (shea, coconut, mineral oil, petrolatum ...) |
| `has_ceramide`, `has_hyaluronic`, `has_squalane` | moisturising ingredients for dry skin |
| `has_fragrance`, `has_alcohol_denat`, `has_essential_oil` | common irritants for sensitive skin |
| `oil_free` | 1 = labelled oil-free (Sephora tag or product name), 0 = not labelled / unknown |

**Concerns written by the website** (1 only when the site itself wrote it)
| Column | Meaning |
|---|---|
| `helps_oil_control`, `helps_acne`, `helps_dryness`, `helps_redness` | the concern the site says the product helps with |
| `helps_with` | the full `helps_with` text, for display |
| `helps_with_source` | `site` = written by the website, `derived` = guessed by our scraper from the ingredients |

**Note:** the `helps_*` columns and the other `tag_*` columns should not be used as inputs when predicting a `tag_*` column.